# Probability of Two Districts Matching Vote Counts

This notebook is a mathematical modeling exercise. It does not try to prove or disprove any claim about an election. The goal is narrower: under explicit probability models, calculate how surprising it is for two fixed districts to have the same vote counts for candidate 1 and candidate 2.

The observed data are:

| district | total votes | candidate 1 | candidate 2 | other / ineligible |
|---|---:|---:|---:|---:|
| Songdo1 | 4548 | 3030 | 1440 | 78 |
| Songdo2 | 4540 | 3030 | 1440 | 70 |

Two different questions are easy to confuse:

- `E_exact`: both districts hit the specific observed candidate counts `(3030, 1440)`.
- `E_collision`: the two districts match each other on candidate 1 and candidate 2 counts, at any common pair `(x, y)`.

This notebook computes both, but the main claim people usually make is about `E_collision` for this fixed district pair.

## Model Choices

For a single district with total votes `N`, model the 3-way count vector as either:

1. **Multinomial plug-in model**
   
   Counts are drawn from `Multinomial(N, p)`, with `p` set to observed local shares. This is an optimistic, data-dependent baseline: it centers each district exactly where the observed result was.

2. **Dirichlet-Multinomial sensitivity model**
   
   First draw `p ~ Dirichlet(alpha)`, then draw counts from `Multinomial(N, p)`. After marginalizing `p`, the count vector follows a Dirichlet-Multinomial distribution. Write `alpha = kappa * u`:

   - `u` is the mean vote-share direction.
   - `kappa` is concentration. Larger `kappa` means districts are tightly concentrated around `u`; smaller `kappa` means more district-to-district heterogeneity.

A key point: choosing `u` from the observed result is not a neutral prior. It is a plug-in / empirical-Bayes-style sensitivity calculation. A defensible prior needs comparable past or neighboring district data.

In [1]:
import numpy as np
from scipy.special import gammaln, logsumexp

# Observed data: categories are candidate 1, candidate 2, and all remaining votes.
N_songdo1 = 4548
N_songdo2 = 4540
observed_c1 = 3030
observed_c2 = 1440

counts_songdo1 = np.array([observed_c1, observed_c2, N_songdo1 - observed_c1 - observed_c2])
counts_songdo2 = np.array([observed_c1, observed_c2, N_songdo2 - observed_c1 - observed_c2])

u_songdo1 = counts_songdo1 / N_songdo1
u_songdo2 = counts_songdo2 / N_songdo2
u_pooled = (counts_songdo1 + counts_songdo2) / (N_songdo1 + N_songdo2)

print("Songdo1 counts:", counts_songdo1, "shares:", np.round(u_songdo1, 6))
print("Songdo2 counts:", counts_songdo2, "shares:", np.round(u_songdo2, 6))
print("Pooled shares:", np.round(u_pooled, 6))

Songdo1 counts: [3030 1440   78] shares: [0.666227 0.316623 0.01715 ]
Songdo2 counts: [3030 1440   70] shares: [0.667401 0.317181 0.015419]
Pooled shares: [0.666813 0.316901 0.016285]


## Exact Probability Functions

The collision probability can be computed exactly using a sequential factorization.

For the multinomial model:

- Candidate 1 count is binomial.
- Conditional on candidate 1 count `x`, candidate 2 count is binomial among the remaining `N - x` votes.

For the Dirichlet-Multinomial model:

- Candidate 1 count is beta-binomial.
- Conditional on candidate 1 count `x`, candidate 2 count is beta-binomial among the remaining `N - x` votes.

This avoids Monte Carlo error. All probability sums are done in log space for numerical stability.

In [2]:
def log_comb(n, k):
    k = np.asarray(k)
    return gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)


def binom_logpmf(k, n, p):
    k = np.asarray(k)
    return log_comb(n, k) + k * np.log(p) + (n - k) * np.log1p(-p)


def betabinom_logpmf(k, n, a, b):
    k = np.asarray(k)
    return (
        log_comb(n, k)
        + gammaln(k + a)
        + gammaln(n - k + b)
        - gammaln(n + a + b)
        + gammaln(a + b)
        - gammaln(a)
        - gammaln(b)
    )


def multinomial2_logpmf(x, y, n, p):
    z = n - x - y
    if x < 0 or y < 0 or z < 0:
        return -np.inf
    return (
        gammaln(n + 1)
        - gammaln(x + 1)
        - gammaln(y + 1)
        - gammaln(z + 1)
        + x * np.log(p[0])
        + y * np.log(p[1])
        + z * np.log(p[2])
    )


def dirichlet_multinomial2_logpmf(x, y, n, alpha):
    z = n - x - y
    if x < 0 or y < 0 or z < 0:
        return -np.inf
    alpha0 = np.sum(alpha)
    return (
        gammaln(n + 1)
        - gammaln(x + 1)
        - gammaln(y + 1)
        - gammaln(z + 1)
        + gammaln(alpha0)
        - gammaln(n + alpha0)
        + gammaln(x + alpha[0])
        - gammaln(alpha[0])
        + gammaln(y + alpha[1])
        - gammaln(alpha[1])
        + gammaln(z + alpha[2])
        - gammaln(alpha[2])
    )


def multinomial_collision_probability(n_a, n_b, p_a, p_b):
    x_max = min(n_a, n_b)
    p_a_y_given_not_x = p_a[1] / (1.0 - p_a[0])
    p_b_y_given_not_x = p_b[1] / (1.0 - p_b[0])

    log_terms = []
    for x in range(x_max + 1):
        y = np.arange(min(n_a - x, n_b - x) + 1)
        log_x = binom_logpmf(x, n_a, p_a[0]) + binom_logpmf(x, n_b, p_b[0])
        log_y = (
            binom_logpmf(y, n_a - x, p_a_y_given_not_x)
            + binom_logpmf(y, n_b - x, p_b_y_given_not_x)
        )
        log_terms.append(log_x + logsumexp(log_y))

    return float(np.exp(logsumexp(log_terms)))


def dirichlet_multinomial_collision_probability(n_a, n_b, alpha_a, alpha_b):
    x_max = min(n_a, n_b)
    log_terms = []
    for x in range(x_max + 1):
        y = np.arange(min(n_a - x, n_b - x) + 1)
        log_x = (
            betabinom_logpmf(x, n_a, alpha_a[0], alpha_a[1] + alpha_a[2])
            + betabinom_logpmf(x, n_b, alpha_b[0], alpha_b[1] + alpha_b[2])
        )
        log_y = (
            betabinom_logpmf(y, n_a - x, alpha_a[1], alpha_a[2])
            + betabinom_logpmf(y, n_b - x, alpha_b[1], alpha_b[2])
        )
        log_terms.append(log_x + logsumexp(log_y))

    return float(np.exp(logsumexp(log_terms)))


def one_in(probability):
    return f"1 in {1.0 / probability:,.0f}"

## Formula Sanity Checks

Before using the formulas on the real vote totals, check them on tiny artificial examples where direct enumeration is cheap. This verifies that the sequential factorization matches the direct joint probability sum.

In [3]:
def direct_multinomial_collision_probability(n_a, n_b, p_a, p_b):
    total = 0.0
    for x in range(min(n_a, n_b) + 1):
        for y in range(min(n_a - x, n_b - x) + 1):
            total += np.exp(multinomial2_logpmf(x, y, n_a, p_a) + multinomial2_logpmf(x, y, n_b, p_b))
    return total


def direct_dm_collision_probability(n_a, n_b, alpha_a, alpha_b):
    total = 0.0
    for x in range(min(n_a, n_b) + 1):
        for y in range(min(n_a - x, n_b - x) + 1):
            total += np.exp(
                dirichlet_multinomial2_logpmf(x, y, n_a, alpha_a)
                + dirichlet_multinomial2_logpmf(x, y, n_b, alpha_b)
            )
    return total


small_n_a, small_n_b = 7, 6
small_p_a = np.array([0.55, 0.35, 0.10])
small_p_b = np.array([0.50, 0.30, 0.20])
small_alpha_a = np.array([8.0, 5.0, 2.0])
small_alpha_b = np.array([7.0, 4.0, 3.0])

mn_direct = direct_multinomial_collision_probability(small_n_a, small_n_b, small_p_a, small_p_b)
mn_factored = multinomial_collision_probability(small_n_a, small_n_b, small_p_a, small_p_b)
dm_direct = direct_dm_collision_probability(small_n_a, small_n_b, small_alpha_a, small_alpha_b)
dm_factored = dirichlet_multinomial_collision_probability(small_n_a, small_n_b, small_alpha_a, small_alpha_b)

assert np.allclose(mn_direct, mn_factored, rtol=1e-12, atol=1e-15)
assert np.allclose(dm_direct, dm_factored, rtol=1e-12, atol=1e-15)

print(f"Multinomial sanity check: direct={mn_direct:.12f}, factored={mn_factored:.12f}")
print(f"Dirichlet-Multinomial sanity check: direct={dm_direct:.12f}, factored={dm_factored:.12f}")

Multinomial sanity check: direct=0.036713151066, factored=0.036713151066
Dirichlet-Multinomial sanity check: direct=0.034489532782, factored=0.034489532782


## Plug-In Multinomial Baseline

This is the most transparent baseline. Each district gets a multinomial probability vector equal to its own observed shares. That makes the observed result sit at the local center of each district's model.

This is useful as an optimistic benchmark, but it is not a pre-data probability. It uses the same observed data to choose the model parameters.

In [4]:
p_collision_plugin = multinomial_collision_probability(
    N_songdo1, N_songdo2, u_songdo1, u_songdo2
)
p_exact_plugin = np.exp(
    multinomial2_logpmf(observed_c1, observed_c2, N_songdo1, u_songdo1)
    + multinomial2_logpmf(observed_c1, observed_c2, N_songdo2, u_songdo2)
)

print("Plug-in multinomial using each district's own observed shares")
print(f"E_collision: {p_collision_plugin:.6e} ({one_in(p_collision_plugin)})")
print(f"E_exact:     {p_exact_plugin:.6e} ({one_in(p_exact_plugin)})")

Plug-in multinomial using each district's own observed shares
E_collision: 2.987566e-04 (1 in 3,347)
E_exact:     3.561539e-07 (1 in 2,807,775)


A slightly less tailored plug-in variant uses one pooled share vector for both districts. This still uses the observed data, but it does not give each district its own separately centered probability vector.

In [5]:
p_collision_pooled = multinomial_collision_probability(
    N_songdo1, N_songdo2, u_pooled, u_pooled
)
p_exact_pooled = np.exp(
    multinomial2_logpmf(observed_c1, observed_c2, N_songdo1, u_pooled)
    + multinomial2_logpmf(observed_c1, observed_c2, N_songdo2, u_pooled)
)

print("Plug-in multinomial using the pooled observed share vector")
print(f"E_collision: {p_collision_pooled:.6e} ({one_in(p_collision_pooled)})")
print(f"E_exact:     {p_exact_pooled:.6e} ({one_in(p_exact_pooled)})")

Plug-in multinomial using the pooled observed share vector
E_collision: 2.413599e-04 (1 in 4,143)
E_exact:     2.878854e-07 (1 in 3,473,604)


## Dirichlet-Multinomial Sensitivity Sweep

Now hold `u` fixed at the observed district shares and sweep `kappa` in `alpha = kappa * u`.

Important interpretation:

- Large `kappa` approaches the multinomial plug-in model.
- Moderate `kappa` allows real district-to-district heterogeneity and usually spreads probability mass over more count vectors.
- Extremely small `kappa` is not realistic for these observed districts. It puts large probability on degenerate outcomes such as nearly all votes in one category. For the broad collision event, those extreme outcomes can create odd behavior, so monotonicity should not be assumed without qualification.

In [6]:
kappas = np.array([0.3, 1, 3, 10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000, 1_000_000], dtype=float)

print("Dirichlet-Multinomial sensitivity with separate observed u vectors")
print(f"{'kappa':>12}  {'P(E_collision)':>16}  {'collision scale':>18}  {'P(E_exact)':>14}  {'exact scale':>18}")
print("-" * 86)

sensitivity_rows = []
for kappa in kappas:
    alpha_1 = kappa * u_songdo1
    alpha_2 = kappa * u_songdo2
    p_collision = dirichlet_multinomial_collision_probability(N_songdo1, N_songdo2, alpha_1, alpha_2)
    p_exact = np.exp(
        dirichlet_multinomial2_logpmf(observed_c1, observed_c2, N_songdo1, alpha_1)
        + dirichlet_multinomial2_logpmf(observed_c1, observed_c2, N_songdo2, alpha_2)
    )
    sensitivity_rows.append((kappa, p_collision, p_exact))
    print(f"{kappa:12,.1f}  {p_collision:16.6e}  {one_in(p_collision):>18}  {p_exact:14.6e}  {one_in(p_exact):>18}")

print("-" * 86)
print(f"{'multinomial':>12}  {p_collision_plugin:16.6e}  {one_in(p_collision_plugin):>18}  {p_exact_plugin:14.6e}  {one_in(p_exact_plugin):>18}")

Dirichlet-Multinomial sensitivity with separate observed u vectors
       kappa    P(E_collision)     collision scale      P(E_exact)         exact scale
--------------------------------------------------------------------------------------


         0.3      5.677789e-05         1 in 17,612    1.318343e-17  1 in 75,852,780,290,749,632


         1.0      6.455580e-06        1 in 154,905    8.993593e-16  1 in 1,111,902,667,935,292


         3.0      1.217194e-06        1 in 821,561    2.783166e-14  1 in 35,930,297,406,156


        10.0      2.963128e-06        1 in 337,481    7.671849e-13  1 in 1,303,466,732,567


...

     3,000.0      1.196263e-04          1 in 8,359    5.619905e-08     1 in 17,793,895


    10,000.0      2.058181e-04          1 in 4,859    1.683149e-07      1 in 5,941,245


    30,000.0      2.596212e-04          1 in 3,852    2.686051e-07      1 in 3,722,937


   100,000.0      2.858244e-04          1 in 3,499    3.258648e-07      1 in 3,068,757


 1,000,000.0      2.974107e-04          1 in 3,362    3.529391e-07      1 in 2,833,350
--------------------------------------------------------------------------------------
 multinomial      2.987566e-04          1 in 3,347    3.561539e-07      1 in 2,807,775


## How to Pick `alpha` From Past Districts

The clean way to choose `alpha` is not to fit it from just these two districts. With only these two districts, and with `u` chosen from the same observed data, fitting concentration is circular.

A better empirical-Bayes workflow is:

1. Collect comparable district-level counts from a past election, or from similar districts in the same election if that is the target population.
2. Estimate `u` from pooled shares across those districts.
3. Estimate `kappa` from how much district shares vary around `u`.

For a Dirichlet-Multinomial with total `N`, the variance of an observed share for one category is approximately:

\[
\operatorname{Var}(X/N) = \frac{u(1-u)(N + \kappa)}{N(\kappa + 1)}.
\]

This matters because observed district shares vary for two reasons:

- ordinary multinomial counting noise, which remains even as `kappa -> infinity`;
- real district-to-district heterogeneity, controlled by finite `kappa`.

The simpler formula `kappa = u(1-u)/s^2 - 1` ignores the finite-count noise floor. It can be okay when `N` is huge and share variation is large, but it is not the right formula for districts with finite vote totals.

In [7]:
def kappa_from_observed_share_sd(u, share_sd, n_effective):
    """Estimate kappa from observed district share SD with finite-N correction.

    The formula solves s^2 = u(1-u)(N+kappa)/(N(kappa+1)).
    It requires the observed variance to be above the multinomial noise floor
    u(1-u)/N and below the maximum Dirichlet-Multinomial share variance u(1-u).
    """
    q = u * (1.0 - u)
    s2 = share_sd**2
    noise_floor = q / n_effective
    if s2 <= noise_floor:
        return np.inf
    if s2 >= q:
        return 0.0
    return n_effective * (q - s2) / (n_effective * s2 - q)


def naive_kappa_from_share_sd(u, share_sd):
    return u * (1.0 - u) / share_sd**2 - 1.0


n_effective = (N_songdo1 + N_songdo2) / 2
candidate_1_share = u_pooled[0]
assumed_sds = [0.007, 0.01, 0.02, 0.03, 0.05]

print("Illustration using candidate 1 share variation across comparable districts")
print(f"Multinomial-only SD floor at N≈{n_effective:.0f}: {np.sqrt(candidate_1_share * (1 - candidate_1_share) / n_effective):.4%}")
print(f"{'assumed share SD':>16}  {'corrected kappa':>16}  {'naive kappa':>14}  {'P(E_collision)':>16}  {'scale':>16}")
print("-" * 84)

for share_sd in assumed_sds:
    kappa_hat = kappa_from_observed_share_sd(candidate_1_share, share_sd, n_effective)
    naive_hat = naive_kappa_from_share_sd(candidate_1_share, share_sd)
    if np.isinf(kappa_hat):
        p_collision = p_collision_plugin
        scale = one_in(p_collision)
        kappa_label = "infinity"
    else:
        alpha_1 = kappa_hat * u_songdo1
        alpha_2 = kappa_hat * u_songdo2
        p_collision = dirichlet_multinomial_collision_probability(N_songdo1, N_songdo2, alpha_1, alpha_2)
        scale = one_in(p_collision)
        kappa_label = f"{kappa_hat:,.1f}"
    print(f"{share_sd:16.2%}  {kappa_label:>16}  {naive_hat:14,.1f}  {p_collision:16.6e}  {scale:>16}")

Illustration using candidate 1 share variation across comparable districts
Multinomial-only SD floor at N≈4544: 0.6992%
assumed share SD   corrected kappa     naive kappa    P(E_collision)             scale
------------------------------------------------------------------------------------


           0.70%       2,090,998.1         4,533.1      2.981114e-04        1 in 3,354


           1.00%           4,345.3         2,220.7      1.467235e-04        1 in 6,816


           2.00%             631.6           554.4      3.776284e-05       1 in 26,481


           3.00%             260.0           245.9      1.769988e-05       1 in 56,498


           5.00%              89.6            87.9      7.836556e-06      1 in 127,607


## Interpretation and Look-Elsewhere Effect

For this fixed pair of districts, the plug-in multinomial collision probability is the easiest baseline to communicate. It answers: if these two districts independently generated votes from probability vectors centered exactly at their observed shares, how often would their candidate 1 and candidate 2 counts match?

That is different from the broader question: among many districts, what is the probability that at least one pair matches on two candidate counts? If there are `D` comparable districts, there are `D(D-1)/2` district pairs. A rough independent-pairs approximation is:

\[
P(\text{at least one collision}) \approx 1 - (1 - p_{pair})^{D(D-1)/2}.
\]

That approximation is not exact because district pairs share districts and because each pair can have different totals and vote-share parameters. But it shows why a fixed-pair probability and an anywhere-in-the-election probability can differ by orders of magnitude.

In [8]:
print("Key fixed-pair results")
print(f"Plug-in multinomial E_collision: {p_collision_plugin:.6e} ({one_in(p_collision_plugin)})")
print(f"Plug-in multinomial E_exact:     {p_exact_plugin:.6e} ({one_in(p_exact_plugin)})")
print()
print("Use these as model-dependent calculations, not universal facts.")
print("A defensible alpha requires comparable district-level data to estimate real share heterogeneity.")

Key fixed-pair results
Plug-in multinomial E_collision: 2.987566e-04 (1 in 3,347)
Plug-in multinomial E_exact:     3.561539e-07 (1 in 2,807,775)

Use these as model-dependent calculations, not universal facts.
A defensible alpha requires comparable district-level data to estimate real share heterogeneity.
